# Семинар 03. Алгоритмы и сложность

Повторяем условия, циклы, функции и списки. На небольших задачах учимся объяснять, как растут время работы и расход памяти. Затем пробуем решать те же задачи быстрее.

## Цели

- написать небольшую функцию по условию и проверить её на примерах;
- выбрать размер входа и посчитать повторяющиеся операции;
- оценить время работы и память своего решения;
- отличить память для результата от памяти для промежуточных данных;
- предложить более быстрый алгоритм и объяснить, почему он работает.

## Перед началом

**Если вы пока осваиваете Python**, начните с заданий 1–3. В них повторяются условия, `for`, `while`, накопление результата и работа со списком. Затем переходите к заданию 4 с перебором пар. Три самостоятельно решённые задачи с объяснением сложности — хороший результат занятия.

**Если небольшие функции вы уже пишете уверенно**, начинайте с задания 5. Оно содержит полное условие и не требует кода из первого блока. Затем выбирайте задания 6 и 7. Они тоже независимы: проходить весь семинар по порядку не требуется.

Во всех заданиях данные передаются через параметры, а ответ возвращается через `return`. Не используйте `input` и `print` внутри решений и не меняйте входные списки. Входные данные соответствуют ограничениям из условия; проверять их типы не нужно.

Сначала разберите пример вручную и опишите решение словами. Замените `raise NotImplementedError` своим кодом, выполните ячейку с функцией, а затем ячейку проверок. `assert` проверяет утверждение: если оно неверно, появится `AssertionError`. Заготовки до решения намеренно не проходят проверки. Добавьте хотя бы один собственный тест к каждой решённой задаче.

## Как оценивать своё решение

После каждой задачи отвечайте на четыре вопроса:

1. Что такое размер входа: длина списка, количество запросов, значение числа?
2. Какую операцию вы считаете и сколько раз она выполняется?
3. Какова оценка времени? Зависит ли число действий только от размера входа или ещё и от значений?
4. Сколько памяти нужно дополнительно? Отдельно назовите память для результата и вспомогательную память.

Считаем сравнение и арифметическую операцию над числами за `O(1)` — это учебное упрощение без учёта стоимости очень больших целых. Индексация списка — `O(1)`, `append` — `O(1)` амортизированно. Операции с одним ключом словаря или элементом множества — `O(1)` в среднем. Стоимость срезов, обходов и встроенных функций учитывайте отдельно.

## Успеет ли решение по времени

В контесте нет строки «требуется `O(n)`»: вместо неё стоит лимит времени, а среди тестов есть большие. Поэтому оценку сложности приходится делать самому — до отправки.

Порядок действий такой: взять из условия максимальные размеры входа, посчитать число операций, сравнить с ориентиром. Для Python считайте, что за секунду выполняется порядка 10⁷–10⁸ простых операций в цикле. Точное число зависит от машины и от того, что именно делается в цикле, поэтому для запаса ориентируйтесь на 10⁷: при лимите 2–3 секунды это несколько десятков миллионов действий.

| Оценка | При `n = 100 000` | Вердикт при лимите 2 с |
|---|---|---|
| `O(n)` | 10⁵ | пройдёт с большим запасом |
| `O(n log n)` | ≈ 1.7 · 10⁶ | пройдёт |
| `O(n √n)` | ≈ 3 · 10⁷ | на грани |
| `O(n²)` | 10¹⁰ | не пройдёт, нужно менять идею |

Ячейка ниже измеряет наивное решение задачи про суммы на отрезках: для каждого запроса заново складываются элементы отрезка. Запустите её и ответьте:

1. Во сколько раз растёт время при удвоении `n` и `q`? Какой оценке это соответствует?
2. Сколько времени займёт `n = q = 100 000`, если продолжить этот рост? Сравните с лимитом 3 секунды.
3. Какую работу этот код повторяет от запроса к запросу?

In [ ]:
import time


def naive_range_sums(values: list[int], queries: list[tuple[int, int]]) -> list[int]:
    result = []
    for left, right in queries:
        total = 0
        for index in range(left, right + 1):
            total += values[index]
        result.append(total)
    return result


for size in (500, 1000, 2000, 4000):
    values = list(range(size))
    queries = [(0, size - 1)] * size
    start = time.perf_counter()
    naive_range_sums(values, queries)
    elapsed = time.perf_counter() - start
    print(f"n = q = {size:5}: {elapsed:.3f} с")

## Разминка. Сначала предскажите, затем запустите

### Что дороже: список или множество

Сколько сравнений в худшем случае делает `target in data` для списка из 200 000 элементов? А `target in lookup` для множества? Предскажите, во сколько раз будет разница, затем запустите.

После запуска ответьте: сколько стоит построение `set(data)` и при каком числе проверок оно окупается? Почему для одной единственной проверки множество строить незачем?

In [ ]:
import time

data = list(range(200_000))
lookup = set(data)
target = 199_999

start = time.perf_counter()
for _ in range(100):
    target in data
list_time = time.perf_counter() - start

start = time.perf_counter()
for _ in range(100):
    target in lookup
set_time = time.perf_counter() - start

print(f"список: {list_time:.4f} с")
print(f"множество: {set_time:.6f} с")

## Часть 1. Задачи

Сначала добейтесь правильного результата, затем разберите число операций и память. Задания 1–3 повторяют цикл с условием, один проход по списку и сужение диапазона. Задание 4 требует идеи: правильного ответа там недостаточно, решение должно укладываться в оценку времени.

### Задание 1. Знаки чисел

Напишите `signs(values)`: функция получает список целых чисел и возвращает **новый список** той же длины. На месте положительного элемента стоит `1`, на месте отрицательного — `-1`, на месте нуля — `0`. Порядок сохраняется, входной список не меняется.

Например, для `[3, -2, 0, 3, -7]` ответ — `[1, -1, 0, 1, -1]`.

Используйте цикл и условие. Готовые функции, которые сами вычисляют знак, использовать нельзя.

In [ ]:
def signs(values: list[int]) -> list[int]:
    raise NotImplementedError

In [ ]:
values = [3, -2, 0, 3, -7]
original = values.copy()
result = signs(values)
assert isinstance(result, list)
assert result == [1, -1, 0, 1, -1]
assert result is not values
assert values == original

assert signs([]) == []
assert signs([0]) == [0]
assert signs([-5, -5]) == [-1, -1]
assert signs([10**9, -10**9]) == [1, -1]

# TODO: добавьте свой тест.

### Задание 2. Самая длинная серия одинаковых

Напишите `longest_run(values)`: функция получает список целых чисел и возвращает длину самой длинной группы подряд идущих равных элементов. Для пустого списка ответ `0`.

Например, в `[5, 5, 1, 1, 1, 2]` самая длинная группа — три единицы, ответ `3`. В `[3, 3, 2, 3]` ответ `2`: две тройки в начале, а последняя тройка стоит отдельно и в группу с ними не объединяется.

Список менять нельзя. Сохранять все группы не нужно — достаточно помнить несколько чисел.

После решения оцените время и вспомогательную память.

In [ ]:
def longest_run(values: list[int]) -> int:
    raise NotImplementedError

In [ ]:
assert longest_run([]) == 0
assert longest_run([7]) == 1
assert longest_run([1, 2, 3]) == 1
assert longest_run([3, 3, 2, 3]) == 2
assert longest_run([5, 5, 1, 1, 1, 2]) == 3
assert longest_run([0, 0, 0]) == 3
assert longest_run([4, 4, 4, 9, 9]) == 3

values = [3, 3, 2, 3]
longest_run(values)
assert values == [3, 3, 2, 3]

# TODO: добавьте свой тест.

### Задание 3. Целочисленный корень

Напишите `int_sqrt(number)`: функция получает неотрицательное целое число и возвращает наибольшее целое `r`, для которого `r * r <= number`. Для `0` ответ равен `0`.

Например, для `15` ответ `3`, потому что `3 * 3 = 9` не превосходит `15`, а `4 * 4 = 16` уже больше. Для `16` ответ `4`.

Нельзя пользоваться `**0.5`, `math.sqrt` и `math.isqrt`. Число доходит до `10**18`, поэтому перебор от нуля не подойдёт: нужно решение за **`O(log number)`**.

После решения ответьте: сколько итераций делает ваш алгоритм для `number = 10**18` и как это число зависит от `number`?

In [ ]:
def int_sqrt(number: int) -> int:
    raise NotImplementedError

In [ ]:
assert int_sqrt(0) == 0
assert int_sqrt(1) == 1
assert int_sqrt(2) == 1
assert int_sqrt(3) == 1
assert int_sqrt(4) == 2
assert int_sqrt(15) == 3
assert int_sqrt(16) == 4
assert int_sqrt(17) == 4
assert int_sqrt(99) == 9
assert int_sqrt(100) == 10
assert int_sqrt(10**18) == 10**9
assert int_sqrt(10**18 - 1) == 10**9 - 1

# TODO: добавьте свой тест.

### Задание 4. Сколько чётных на отрезке

Напишите `count_even_in_ranges(values, queries)`: функция получает список целых чисел и список запросов. Каждый запрос — пара `(left, right)`, индексы считаются с нуля, **обе границы включаются**. Для каждого запроса верните количество чётных чисел на этом отрезке; ответы идут в порядке запросов.

Например, для `values = [3, 8, 0, -4, 7]` и запроса `(1, 2)` ответ `2`: чётные `8` и `0`. Запросы могут повторяться. Если запросов нет, верните пустой список. Ноль и отрицательные чётные числа учитываются.

Размеры: до `100_000` элементов и до `100_000` запросов. Нужно уложиться в **`O(n + q)`** — вся подготовка плюс все ответы. Входные списки менять нельзя.

Здесь недостаточно правильного ответа: решение должно укладываться в оценку времени. Если быстрое не получается сразу, напишите простое правильное, найдите, какую работу оно повторяет, и уберите это повторение.

Перед кодом подумайте: сколько операций сделает решение, которое на каждый запрос заново проходит отрезок, при `n = q = 100_000`?

In [ ]:
def count_even_in_ranges(
    values: list[int], queries: list[tuple[int, int]]
) -> list[int]:
    raise NotImplementedError

In [ ]:
values = [3, 8, 0, -4, 7]
queries = [(0, 4), (1, 2), (4, 4), (0, 0), (1, 2)]
original_values = values.copy()
original_queries = queries.copy()
result = count_even_in_ranges(values, queries)
assert result == [3, 2, 0, 0, 2]
assert values == original_values
assert queries == original_queries

assert count_even_in_ranges([], []) == []
assert count_even_in_ranges([2], [(0, 0)]) == [1]
assert count_even_in_ranges([1], [(0, 0)]) == [0]
assert count_even_in_ranges([1, 3, 5], [(0, 2)]) == [0]
assert count_even_in_ranges([2, 4, 6], [(0, 2), (1, 1)]) == [3, 1]
assert count_even_in_ranges([1, 2, 3], []) == []

# TODO: добавьте свой тест.

## Часть 2. Задачи на идею

Эти два задания независимы друг от друга и от первой части. Правильного ответа здесь недостаточно: решение должно укладываться в требуемую оценку, а вы — объяснить, почему оно не пропускает нужные варианты и не считает лишние.

Если быстрое решение не получается сразу, напишите простое правильное. Затем найдите, какую работу оно повторяет, и уберите это повторение.

### Задание 5. Равные пары в большом списке

Напишите `count_equal_pairs_fast(values)`: функция получает список целых чисел и возвращает количество пар позиций `i < j`, для которых `values[i] == values[j]`. Пары считаются по позициям, поэтому одинаковых значений может быть сколько угодно.

Например, для `[5, 2, 5, 5]` ответ равен `3`, для `[7, 7, 7, 7]` — `6`, а для пустого списка — `0`.

В списке до `100_000` элементов. Добейтесь **`O(n)` времени в среднем**, где `n` — длина списка. Исходный список менять нельзя.

Объясните, как ваше решение учитывает каждую подходящую пару ровно один раз. От чего зависит расход памяти? На каких данных он наибольший? Сравните сложность с перебором всех пар.

In [ ]:
def count_equal_pairs_fast(values: list[int]) -> int:
    raise NotImplementedError

In [ ]:
assert count_equal_pairs_fast([]) == 0
assert count_equal_pairs_fast([5]) == 0
assert count_equal_pairs_fast([1, 2, 3]) == 0
assert count_equal_pairs_fast([5, 2, 5, 5]) == 3
assert count_equal_pairs_fast([7, 7, 7, 7]) == 6
assert count_equal_pairs_fast([-1, 0, -1, 0]) == 2

values = [5, 2, 5, 5]
count_equal_pairs_fast(values)
assert values == [5, 2, 5, 5]

# TODO: добавьте свой тест.

### Задание 6. Лучшая сумма подряд

Напишите `max_block_sum(values, k)`: функция получает непустой список целых чисел и целое `k`, где `1 <= k <= len(values)`. Верните наибольшую сумму **ровно `k` соседних элементов**.

Например, у `[2, 1, 5, 1, 3, 2]` при `k = 3` есть четыре подходящих отрезка с суммами `8`, `7`, `9` и `6`. Ответ — `9`.

Числа могут быть отрицательными. Выбирать меньше `k` элементов или пропускать элементы внутри отрезка нельзя. Для `[-8, -2, -5]` и `k = 2` ответ равен `-7`.

Добейтесь **`O(n)` времени и `O(1)` вспомогательной памяти**, где `n` — длина списка. `k` может быть любым допустимым, в том числе расти вместе с `n`.

Объясните, почему рассмотрены все отрезки нужной длины. Оцените также простое решение, которое заново складывает каждый отрезок: как его время зависит от `n` и `k`?

In [ ]:
def max_block_sum(values: list[int], k: int) -> int:
    raise NotImplementedError

In [ ]:
assert max_block_sum([2, 1, 5, 1, 3, 2], 3) == 9
assert max_block_sum([-8, -2, -5], 2) == -7
assert max_block_sum([-8, -2, -5], 1) == -2
assert max_block_sum([4, -1, 2], 3) == 5
assert max_block_sum([5], 1) == 5
assert max_block_sum([0, 0, 0], 2) == 0
assert max_block_sum([9, 8, -10, -10], 2) == 17
assert max_block_sum([-10, -10, 8, 9], 2) == 17

values = [2, 1, 5, 1, 3, 2]
max_block_sum(values, 3)
assert values == [2, 1, 5, 1, 3, 2]

# TODO: добавьте свой тест.

## Итоги

- Условия, циклы и функции позволяют решать задачи и одновременно разбирать сложность своего кода.
- Оценка зависит от размера входа и количества действий, а не от числа строк программы.
- Память для результата и вспомогательная память — разные части оценки.
- Ускорение часто начинается с вопроса: какую работу алгоритм повторяет и можно ли её не повторять?

Для каждой решённой задачи оставьте код, свои проверки и короткое объяснение сложности. Если решали второй блок, добавьте объяснение корректности выбранной идеи.

[Домашка](https://contest.yandex.ru/contest/100166/enter/?retPage=).